In [ ]:
using ProgressBars
include("/Users/travail/Desktop/Quantic/Dev/Stage_quantic_2025/Code_Julia/Julia_file/classical_oscillators.jl");

In [ ]:
# Parameters

const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

ω_t=5.6*2*pi
ω_a=7.5*2*pi
E_Ct=0.280*2*pi
C_t_inv=2*E_Ct # hbar/e^2 GHz
E_J = 50*E_Ct
ϵ_d=0.05
ω_d=ω_a
g=0.25*2*pi

κ_a = 0.1

C_a_inv=0.8859291283123218 #copy of C_b_inv for the ATS circuit
EC_a=C_a_inv/2;

In [ ]:
EC_a/2pi

In [ ]:
sqrt(4*E_J*C_t_inv)/2pi

In [ ]:
function readout_1d_chain(n, t_max, chain_params;
     circuit_params=(ω_t=5.304*2*pi,ω_a=7.5*2*pi, E_Ct=0.280*2*pi, E_Ca=0.44, E_J = 50*E_Ct, ϵ_d=0.22, ω_d=ω_a, g=0.25*2*pi),
     init_transmon=(ϕ_t_0=0.0,q_t_0 = 0.0), init_memory=(ϕ_a_0=0.0,q_a_0 = 0.0), emission=false, fast_no_chain=true)



    # Intialization of the circuit parameters
    ω_t, ω_a, E_Ct, E_Ca, E_J, ϵ_d, ω_d, g = circuit_params
    C_t_inv=2*E_Ct # hbar/e^2 GHz
    Z_t=C_t_inv/ω_t # Impedance of the tranmson
    C_a_inv=2*E_Ca # hbar/e^2 GHz
    Z_a=C_a_inv/ω_a # Impedance of the memory

    ϕ_quantum=1/2 #flux quantum in hbar/e

    onsite_energies, hopping_energies, system_chain_coupling = chain_params

    function drive(t)
        return ϵ_d * cos(ω_d * t) #We consider a drive resonant with the mode b
    end

    temp_buf = zeros(n)
    function readout_ODE(ddu,du,u,p,t)
        ϕ_t = u[1]
        ϕ_a = u[2]
        Φ=u[3:end]

        ddu[1] = -ω_t*E_J*sqrt(Z_t) * sin(ϕ_t*sqrt(Z_t)/ϕ_quantum) / ϕ_quantum - ω_a * g * sqrt(2/Z_t) * ϕ_a
        ddu[2] = -ω_a^2*ϕ_a - g*sqrt(2)*E_J*sin(ϕ_t * sqrt(Z_t)/ϕ_quantum) / ϕ_quantum + ω_d * ϵ_d * sqrt(2)*cos(ω_d * t) - sqrt(Z_a)*system_chain_coupling *(onsite_energies[1]*Φ[1] + hopping_energies[1]*Φ[2])
        
        mul!(temp_buf, Mϕ_chain, Φ)
        @. temp_buf = -temp_buf
        mul!(view(ddu, 3:length(ddu)), Mq_chain, temp_buf)
        ddu[3] -= system_chain_coupling * ω_a * ϕ_a * sqrt(Z_a)
        return nothing
    end


    gq=vcat(- hopping_energies)
    gϕ=vcat(hopping_energies)

    Mq_chain=SymTridiagonal(onsite_energies, gq)
    Mϕ_chain=SymTridiagonal(onsite_energies, gϕ)

    Mq_full=SymTridiagonal(vcat([ω_t,ω_a], onsite_energies), vcat([g/sqrt(2*Z_t),system_chain_coupling/2],gq))

    #Initial conditions
    ϕ_t_0,q_t_0 = init_transmon

    ϕ_a_0,q_a_0 = init_memory


    ϕ_0 = vcat(ϕ_t_0/sqrt(Z_t),ϕ_a_0/sqrt(Z_a), fill(0.0, n))
    q_0 = vcat(q_t_0 * sqrt(Z_t),q_a_0 * sqrt(Z_a), fill(0.0, n))
    ϕ_dot_0 = Mq_full*q_0


    #Run the simulation
    tspan = (0.0, t_max) # Time span for the simulation
    dt = 1 / (5 * maximum(vcat(ω_t,ω_a,onsite_energies))) # Time step for the simulation
    prob = SecondOrderODEProblem(readout_ODE, ϕ_dot_0, ϕ_0, tspan)

    if emission
        save_idxs=vcat(3:n+2,n+5:n+4+n) # We save the chain DoFs (Φ)
        sol = solve(prob, VelocityVerlet(),dt=dt,
        saveat=t_max,                      # how often to save (or choose a coarser step)
        save_idxs=save_idxs,            # only store system DoFs
        save_everystep=false            # do not store intermediate solver steps
        )
        ϕ_dot = [getindex.(sol.u,i)[end] for i in 1:n]
        ϕ = [getindex.(sol.u,i+n)[end] for i in 1:n]
        return(ϕ,ϕ_dot)
    
    end

    if fast_no_chain
        save_idxs=[1,2,n+3,n+4] # We only save the system DoFs (ϕ_a, ϕ_b) and not the chain DoFs (Φ)
        sol = solve(prob, VelocityVerlet(),dt=dt,
        saveat=dt,                      # how often to save (or choose a coarser step)
        save_idxs=save_idxs,            # only store system DoFs
        save_everystep=false            # do not store intermediate solver steps
        )

        ϕ_dot = [getindex.(sol.u,i) for i in 1:2]
        ϕ = [getindex.(sol.u,i+2) for i in 1:2]
    
        ϕ_t = ϕ[1] * sqrt(Z_t)
        q_t = ϕ_dot[1]/ω_t / sqrt(Z_t)

        ϕ_a = ϕ[2] * sqrt(Z_a)
        q_a = ϕ_dot[2]/ω_a / sqrt(Z_a)

        t_list = sol.t
        return(ϕ_t,ϕ_a,q_t,q_a,t_list)
    end
    
    sol = solve(prob, VelocityVerlet(),dt=dt)
    
    ϕ_dot = [getindex.(sol.u,i) for i in 1:n+2]
    ϕ = [getindex.(sol.u,i+n+2) for i in 1:n+2]
    ϕ_t = ϕ[1] * sqrt(Z_t)
    ϕ_a = ϕ[2] * sqrt(Z_a)

    fac = lu(Mq_full)
    q = [fac \ getindex.(ϕ_dot,t) for t in 1:length(sol.t)]
    q = [getindex.(q,i) for i in 1:n+2]
    q_t = q[1] / sqrt(Z_t)
    q_a = q[2] / sqrt(Z_a)

    t_list = sol.t
    return(ϕ_t,ϕ_a,ϕ[3:end],q_t,q_a,q[3:end],t_list)
end

In [ ]:
n=3000

bath=Exp_decay
bath_params =(ω_0=50*2*pi, ω_cut_off=20*2*pi)

AB, J = bath(bath_params=bath_params,κ=κ_a, C_inv=C_a_inv)
chain_params = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=7000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

In [ ]:
t_max = 13/κ_a
ϕ_t,ϕ_a,q_t,q_a,t_list = readout_1d_chain(n, t_max, chain_params;
     circuit_params=(ω_t=ω_t,ω_a=ω_a, E_Ct=E_Ct, E_Ca=E_Ca, E_J = E_J, ϵ_d=0, ω_d=ω_d, g=0),
     init_transmon=(ϕ_t_0=.0,q_t_0 = 0.0), init_memory=(ϕ_a_0=1.0,q_a_0 = 0.0), emission=false, fast_no_chain=true)

In [ ]:
plot(t_list*κ_a,ϕ_a)

In [ ]:
F,freqs=FFT_perso(ϕ_a, t_list)
f_shifted=abs(freqs[argmax(abs.(F))])
plot(freqs,abs.(F), label="")
vline!([f_shifted], linestyle=:dash, color=:red, label="f_shifted = $(round(f_shifted, digits=3)) GHz")
plot!(xlims=(4,8))

In [ ]:
t_max = 13/κ_a
ϕ_t,ϕ_a,q_t,q_a,t_list = readout_1d_chain(n, t_max, chain_params;
     circuit_params=(ω_t=ω_t,ω_a=ω_a, E_Ct=E_Ct, E_Ca=E_Ca, E_J = E_J, ϵ_d=0, ω_d=7.625*2*pi, g=g),
     init_transmon=(ϕ_t_0=1.0,q_t_0 = 0.0), init_memory=(ϕ_a_0=1.0,q_a_0 = 0.0), emission=false, fast_no_chain=true)

In [ ]:
plot(t_list*κ_a,ϕ_a)
plot!(xlabel=L"\kappa_a t", ylabel=L"\phi_a(t)", title="Dissipation process with no drive", legend=:topright, label="")
savefig("../../Figures/Readout/dissipation_process_no_drive.pdf")

In [ ]:
F,freqs=FFT_perso(ϕ_a, t_list)
f_shifted=abs(freqs[argmax(abs.(F))])
plot(freqs,abs.(F), label="")
vline!([f_shifted], linestyle=:dash, color=:red, label="f_shifted= $(round(f_shifted, digits=3)) GHz")
plot!(xlims=(5,9), yaxis=:log10, legend=:bottomright, xlabel=L"f (GHz)", ylabel=L"|F(f)|", title="Fourier transform of the phase")
#savefig("../../Figures/Readout/Fourier_transform_phase_dissipation_no_drive.pdf")

In [ ]:
t_max = 13/κ_a
ω_d=f_shifted*2*pi
ϕ_t,ϕ_a,q_t,q_a,t_list = readout_1d_chain(n, t_max, chain_params;
     circuit_params=(ω_t=ω_t,ω_a=ω_a, E_Ct=E_Ct, E_Ca=E_Ca, E_J = E_J, ϵ_d=ϵ_d, ω_d=ω_d, g=g),
     init_transmon=(ϕ_t_0=0.0,q_t_0 = 0.0), init_memory=(ϕ_a_0=0.0,q_a_0 = 0.0), emission=false, fast_no_chain=true)

In [ ]:
plot(t_list*κ_a,ϕ_a)

In [ ]:
plot(t_list*κ_a,ϕ_t)

In [ ]:
F,freqs=FFT_perso(ϕ_a, t_list)
f_shifted=abs(freqs[argmax(abs.(F))])
plot(freqs,abs.(F))
vline!([f_shifted], linestyle=:dash, color=:red, label="f_shifted")
vline!([7.625], linestyle=:dash, color=:red, label="f_shifted")
plot!(xlims=(4,8))

In [ ]:
F,freqs=FFT_perso(ϕ_t, t_list)
f_shifted=abs(freqs[argmax(abs.(F))])
plot(freqs,abs.(F))
vline!([f_shifted], linestyle=:dash, color=:red, label="f_shifted")
vline!([7.625], linestyle=:dash, color=:red, label="")
vline!([5.42], linestyle=:dash, color=:red, label="")
plot!(xlims=(4,8))

In [ ]:
t_max = 13/κ_a
ϕ_t,ϕ_a,q_t,q_a,t_list = readout_1d_chain(n, t_max, chain_params;
     circuit_params=(ω_t=ω_t,ω_a=ω_a, E_Ct=E_Ct, E_Ca=E_Ca, E_J = E_J, ϵ_d=ϵ_d, ω_d=ω_d, g=g),
     init_transmon=(ϕ_t_0=0.0,q_t_0 = 0.0), init_memory=(ϕ_a_0=0.0,q_a_0 = 0.0), emission=false, fast_no_chain=true)

In [ ]:
plot(t_list*κ_a,ϕ_a)

In [ ]:
plot(t_list*κ_a,ϕ_t)

In [ ]:
F,freqs=FFT_perso(ϕ_t, t_list)
f_shifted_bis=abs(freqs[argmax(abs.(F))])
plot(freqs,abs.(F))
vline!([f_shifted_bis], linestyle=:dash, color=:red, label="f_shifted")
plot!(xlims=(4,8))

In [ ]:
t_max = 13/κ_a
ϕ_t,ϕ_a,q_t,q_a,t_list = readout_1d_chain(n, t_max, chain_params;
     circuit_params=(ω_t=ω_t,ω_a=ω_a, E_Ct=E_Ct, E_Ca=E_Ca, E_J = E_J, ϵ_d=0, ω_d=ω_d, g=0),
     init_transmon=(ϕ_t_0=1.0,q_t_0 = 0.0), init_memory=(ϕ_a_0=0.0,q_a_0 = 0.0), emission=false, fast_no_chain=true)

In [ ]:
plot(t_list*κ_a,ϕ_t)

In [ ]:
F,freqs=FFT_perso(ϕ_t, t_list)
f_shifted_bis=abs(freqs[argmax(abs.(F))])
plot(freqs,abs.(F))
vline!([f_shifted_bis], linestyle=:dash, color=:red, label="f_shifted")
plot!(xlims=(0,8))

In [ ]:
gq=vcat(- chain_params[2])
gϕ=vcat(chain_params[2])

Mq_chain=SymTridiagonal(chain_params[1], gq)
Mϕ_chain=SymTridiagonal(chain_params[1], gϕ)

A_chain = Mq_chain * Mϕ_chain

vals_chain, vecs_chain = eigen(A_chain)
ωk²_chain = real(vals_chain)
ωk_chain = sqrt.(ωk²_chain)
P_chain = real(vecs_chain)


Dq_chain = diag(P_chain'*(Mq_chain \ P_chain))  # uses linear solve instead of inv
Dϕ_chain = diag(P_chain' * Mϕ_chain * P_chain);

In [ ]:
ϕ_chain,ϕ_dot_chain = readout_1d_chain(n, t_max, chain_params;
     circuit_params=(ω_t=ω_t,ω_a=ω_a, E_Ct=E_Ct, E_Ca=E_Ca, E_J = E_J, ϵ_d=ϵ_d, ω_d=ω_d, g=g),
     init_transmon=(ϕ_t_0=0.0,q_t_0 = 0.0), init_memory=(ϕ_a_0=0.0,q_a_0 = 0.0), emission=true)

In [ ]:
# Solve P_chain * S = ϕ for S instead of inverting
S = P_chain \ ϕ_chain
S_dot = P_chain \ ϕ_dot_chain

# Energy in normal modes
Energies_normal_modes_chain = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2;

In [ ]:
plot(ωk_chain[10:end-1]/2pi,Energies_normal_modes_chain[10:end-1],label="")
plot!(ylabel="Energy stored in the chain", xlabel="Frequency (GHz)", title = L"κ_at = "*"$(κ_a*t_max), "*L"n_{osc}="*"$(n), "*L"\varepsilon_d="*"$(ϵ_d) GHz", legend=:bottomright, yaxis=:log)
vline!([5.42], linestyle=:dash, color=:red, label=L"\omega_t"* "= $(round(ω_t/2pi, digits=3)) GHz")
vline!([7.625], linestyle=:dash, color=Perso_palette_hex[4], label=L"\omega_a^{shifted}"* "= $(round(f_shifted, digits=3)) GHz")
#savefig("../../Figures/Readout/normal_modes_energy_chain_example.pdf")

In [ ]:
(ϵ_d/(κ_a/2))^2

In [ ]:
n_list = 1:10
Energies_list = Array{Float64}(undef, length(n_list), length(ωk_chain))
for (i,n_i) in ProgressBar(enumerate(n_list))
    t_max = 13/κ_a
    ϵ_d_i = κ_a/2 * sqrt(n_i)
    ϕ_chain,ϕ_dot_chain = readout_1d_chain(n, t_max, chain_params;
     circuit_params=(ω_t=ω_t,ω_a=ω_a, E_Ct=E_Ct, E_Ca=E_Ca, E_J = E_J, ϵ_d=ϵ_d_i, ω_d=ω_d, g=g),
     init_transmon=(ϕ_t_0=0.0,q_t_0 = 0.0), init_memory=(ϕ_a_0=0.0,q_a_0 = 0.0), emission=true)
    
    S = P_chain \ ϕ_chain
    S_dot = P_chain \ ϕ_dot_chain

    # Energy in normal modes
    Energies_normal_modes_chain = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2;
    Energies_list[i,:] = Energies_normal_modes_chain
end

In [ ]:
heatmap(ωk_chain[10:end-1]/2pi,n_list, log10.(Energies_list[:,10:end-1]))
plot!( xlabel="Frequency (GHz)", ylabel=L"\bar n", title = L"κ_at = "*"$(κ_a*t_max), "*L"n_{osc}="*"$(n)", colorbar_title="\n"*"Energy stored in the chain (log scale)",
right_margin=20mm)
#savefig("../../Figures/Readout/normal_modes_energy_chain_vs_n.pdf")

In [ ]:
freqs_int_1=argmin(abs.(ωk_chain[:]/2pi .- 5.42))
freqs_int_2=argmin(abs.(ωk_chain[:]/2pi .- f_shifted))
freqs_int_3=argmin(abs.(ωk_chain[:]/2pi .- 7.625))

plot(n_list, Energies_list[:,freqs_int_1]/Energies_list[1,freqs_int_1], label="peaks at 5.42 GHz")
plot!(n_list, Energies_list[:,freqs_int_2]/Energies_list[1,freqs_int_2], label="peaks at the shifted frequency")
plot!(n_list, Energies_list[:,freqs_int_3]/Energies_list[1,freqs_int_3], label="peaks at 7.625 GHz")
plot!(xlabel=L"\bar n", ylabel="Normalised energy stored in the chain", title = L"κ_at = "*"$(κ_a*t_max), "*L"n_{osc}="*"$(n)")

#savefig("../../Figures/Readout/normal_modes_energy_chain_vs_n_peaks_evolution.pdf")